In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from scipy.stats import ks_2samp, anderson_ksamp

import warnings
import tqdm

## Reading samples

### HARPS

In [17]:
harps_c = pd.read_csv('harps_gaia_c.csv')
harps_mg_si = pd.read_csv('harps_gaia_mg_si.csv')
harps_o = pd.read_csv('harps_gaia_o.csv')

print (f'harps_c contains {len(harps_c)} stars')
print (f'harps_mg_si contains {len(harps_mg_si)} stars')    
print (f'harps_o contains {len(harps_o)} stars')

# Ensure 'source_id' is int and set as index
for df in [harps_mg_si, harps_c, harps_o]:
    df['source_id'] = df['source_id'].astype(int)
    df.set_index('source_id', inplace=True)

# Start with the base DataFrame
harps_combined = harps_mg_si.copy()

# Add only new columns from harps_c
new_cols_c = harps_c.columns.difference(harps_combined.columns)
harps_combined = harps_combined.join(harps_c[new_cols_c], how='left')

# Add only new columns from harps_o
new_cols_o = harps_o.columns.difference(harps_combined.columns)
harps_combined = harps_combined.join(harps_o[new_cols_o], how='left')

# Bring source_id back as a column
harps_combined.reset_index(inplace=True)
harps_combined.to_csv('harps_sample.csv')

harps_combined.head(2)

harps_c contains 627 stars
harps_mg_si contains 798 stars
harps_o contains 232 stars


,source_id,Star,ra,dec,parallax,parallax_error,pmra,pmra_error,pmdec,pmdec_error,...,fe_h,e_fe_h,mg_h,e_mg_h,si_h,e_si_h,c_h,e_c_h,e_o_h,o_h
0,3790755044736796544,BD-012505,169.304406,-1.981918,30.138378,0.023513,-568.299830,0.017789,-14.309893,0.014746,...,-0.07,0.02,-0.13,0.10,0.09,0.09,NaN,NaN,NaN,NaN
1,4706630501049679744,HD55,1.322329,-67.835081,61.006400,0.019343,-116.617908,0.019904,-560.587881,0.018929,...,-0.67,0.01,-0.55,0.07,-0.46,0.08,NaN,NaN,NaN,NaN


### APOGEE

In [18]:
apogee = pd.read_csv('apogeeDR17_MST_final_sample.csv')

# Mapping from apogee column names to harps column names
apogee_rename_map = {
    'RA': 'ra',
    'DEC': 'dec',
    'GAIAEDR3_SOURCE_ID': 'source_id',
    'GAIAEDR3_PARALLAX': 'parallax',
    'GAIAEDR3_PARALLAX_ERROR': 'parallax_error',
    'GAIAEDR3_PMRA': 'pmra',
    'GAIAEDR3_PMRA_ERROR': 'pmra_error',
    'GAIAEDR3_PMDEC': 'pmdec',
    'GAIAEDR3_PMDEC_ERROR': 'pmdec_error',
    'GAIAEDR3_PHOT_G_MEAN_MAG': 'phot_g_mean_mag',
    'GAIAEDR3_PHOT_BP_MEAN_MAG': 'phot_bp_mean_mag',
    'GAIAEDR3_PHOT_RP_MEAN_MAG': 'phot_rp_mean_mag',
    'GAIAEDR3_R_MED_PHOTOGEO': 'distance_gspphot',  
    'TEFF': 'Teff',
    'TEFF_ERR': 'e_Teff',
    'LOGG': 'logg',
    'LOGG_ERR': 'e_logg',
    'CI_H': 'c_h', # this is atomic carbon abundance
    'CI_H_ERR': 'e_c_h',
    'O_H': 'o_h',
    'O_H_ERR': 'e_o_h',
    'MG_H': 'mg_h',
    'MG_H_ERR': 'e_mg_h',
    'SI_H': 'si_h',
    'SI_H_ERR': 'e_si_h',
    'FE_H': 'fe_h',
    'FE_H_ERR': 'e_fe_h',
    'VHELIO_AVG': 'radial_velocity',
    'VERR': 'radial_velocity_error',
    'U_lsr': 'U_lsr',
    'V_lsr': 'V_lsr',
    'W_lsr': 'W_lsr',
    'P_gal': 'P_gal'
}

# Rename apogee columns
apogee_renamed = apogee.rename(columns=apogee_rename_map)
apogee_renamed.drop(['Unnamed: 0'], axis = 1, inplace = True)
apogee_renamed.to_csv('apogee_sample.csv', index=False)
apogee_renamed.head(2)

,ra,dec,GLON,GLAT,radial_velocity,radial_velocity_error,source_id,parallax,parallax_error,pmra,...,mg_h,e_mg_h,si_h,e_si_h,fe_h,e_fe_h,U_lsr,V_lsr,W_lsr,P_gal
0,0.005558,57.354549,116.014776,-4.824917,-101.71,0.07,421086363305436800,0.13,0.03,-1.15,...,-0.38,0.02,-0.42,0.03,-0.51,0.01,-89.86,-60.07,37.45,19.2
1,0.008802,63.463078,117.227985,1.162167,-106.63,0.03,431594980053422720,0.07,0.04,-0.92,...,-0.06,0.01,-0.09,0.02,-0.08,0.01,-125.17,-47.86,-43.08,5.1


### Galah

In [19]:
galah = pd.read_csv('galahDR4_final_sample.csv')
galah.drop(columns = ['Unnamed: 0', 'flag_c_fe'], axis = 1, inplace = True)
galah_c = pd.read_csv('galahDR4_final_carbon_sample.csv')

# Ensure both have the same type for the identifier
galah['gaiadr3_source_id'] = galah['gaiadr3_source_id'].astype(int)
galah_c['gaiadr3_source_id'] = galah_c['gaiadr3_source_id'].astype(int)

# Create a set of IDs from the cleaned DF
cleaned_ids = set(galah_c['gaiadr3_source_id'])

# Add the flag column to galah
galah['c_flag'] = galah['gaiadr3_source_id'].isin(cleaned_ids)

# Mapping from galah column names to harps-style column names
galah_rename_map = {
    'gaiadr3_source_id': 'source_id',
    'rv_comp_1': 'radial_velocity',
    'e_rv_comp_1': 'radial_velocity_error',
    'teff': 'Teff',
    'e_teff': 'e_Teff',
    'logg': 'logg',
    'e_logg': 'e_logg',
    'fe_h': 'fe_h',
    'e_fe_h': 'e_fe_h',
    'c_h': 'c_h',
    'e_c_h': 'e_c_h',
    'o_h': 'o_h',
    'e_o_h': 'e_o_h',
    'mg_h': 'mg_h',
    'e_mg_h': 'e_mg_h',
    'si_h': 'si_h',
    'e_si_h': 'e_si_h',
    'ra': 'ra',
    'dec': 'dec',
    'parallax': 'parallax',
    'parallax_error': 'parallax_error',
    'pmra': 'pmra',
    'pmra_error': 'pmra_error',
    'pmdec': 'pmdec',
    'pmdec_error': 'pmdec_error',
    'phot_g_mean_mag': 'phot_g_mean_mag',
    'phot_bp_mean_mag': 'phot_bp_mean_mag',
    'phot_rp_mean_mag': 'phot_rp_mean_mag',
    'r_med_photogeo': 'distance_gspphot',  # rename to harps-style
    'U_lsr': 'U_lsr',
    'V_lsr': 'V_lsr',
    'W_lsr': 'W_lsr',
    'P_gal': 'P_gal'
}

# Apply the renaming
galah_renamed = galah.rename(columns=galah_rename_map)
galah_renamed.to_csv('galah_sample.csv', index = False)